In [ ]:

# 1) setup

%run "./03_preprocessing.ipynb"

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

required_objects = [
    "X_train",
    "y_train",
    "X_test"
]

for obj in required_objects:
    if obj not in globals():
        raise NameError(
            f"{obj} does not exist. Run Notebook 3 before running this notebook."
        )

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# 2) basic feature quality check

feature_quality = pd.DataFrame({
    "missing_values": X_train.isna().sum(),
    "unique_values": X_train.nunique(),
    "dtype": X_train.dtypes.astype(str)
})

feature_quality = feature_quality.sort_values(
    "unique_values",
    ascending=True
)

print("features:", X_train.shape[1])
print("missing values in X_train:", X_train.isna().sum().sum())
print("non-numeric columns:", len(X_train.select_dtypes(exclude="number").columns))

display(feature_quality.head(20))

In [ ]:
# 3) constant and near-constant features

near_constant_threshold = 0.99

constant_features = X_train.columns[
    X_train.nunique() <= 1
].tolist()

near_constant_features = []

for col in X_train.columns:
    most_common_ratio = X_train[col].value_counts(normalize=True).iloc[0]

    if most_common_ratio >= near_constant_threshold:
        near_constant_features.append(col)

near_constant_features = sorted(set(near_constant_features))

print("constant features:", len(constant_features))
print("near-constant features:", len(near_constant_features))

display(
    pd.DataFrame({
        "feature": near_constant_features
    }).head(30)
)

In [ ]:
# 4) correlation check

correlation_threshold = 0.95

corr_matrix = X_train.corr(numeric_only=True).abs()

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
    .reset_index()
)

high_corr_pairs.columns = [
    "feature_1",
    "feature_2",
    "correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["correlation"] >= correlation_threshold
]

print("highly correlated feature pairs:", len(high_corr_pairs))

display(high_corr_pairs.head(30))

In [ ]:
# 5) choose correlated features to remove

features_to_remove_from_correlation = set()

for _, row in high_corr_pairs.iterrows():
    feature_1 = row["feature_1"]
    feature_2 = row["feature_2"]

    if feature_1 in features_to_remove_from_correlation:
        continue

    if feature_2 in features_to_remove_from_correlation:
        continue

    unique_1 = X_train[feature_1].nunique()
    unique_2 = X_train[feature_2].nunique()

    if unique_1 <= unique_2:
        features_to_remove_from_correlation.add(feature_1)
    else:
        features_to_remove_from_correlation.add(feature_2)

features_to_remove_from_correlation = sorted(
    features_to_remove_from_correlation
)

print("features selected for removal due to correlation:", len(features_to_remove_from_correlation))

display(
    pd.DataFrame({
        "feature": features_to_remove_from_correlation
    }).head(30)
)

In [ ]:
# 6) train/validation split for feature importance

X_train_part, X_valid_part, y_train_part, y_valid_part = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("X_train_part shape:", X_train_part.shape)
print("X_valid_part shape:", X_valid_part.shape)
print("y_train_part shape:", y_train_part.shape)
print("y_valid_part shape:", y_valid_part.shape)

In [ ]:
# 7) feature importance from baseline model

importance_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

importance_model.fit(
    X_train_part,
    y_train_part
)

valid_predictions = importance_model.predict(X_valid_part)

valid_accuracy = accuracy_score(
    y_valid_part,
    valid_predictions
)

print("validation accuracy of feature-importance model:", round(valid_accuracy, 4))

In [ ]:
# 8) display feature importance

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": importance_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

display(feature_importance.head(30))

In [ ]:
# 9) select final features

zero_importance_features = (
    feature_importance[
        feature_importance["importance"] == 0
    ]["feature"]
    .tolist()
)

features_to_remove = sorted(
    set(near_constant_features)
    | set(features_to_remove_from_correlation)
    | set(zero_importance_features)
)

selected_features = [
    col for col in X_train.columns
    if col not in features_to_remove
]

print("original features:", X_train.shape[1])
print("features removed:", len(features_to_remove))
print("selected features:", len(selected_features))

removed_features_summary = pd.DataFrame({
    "feature": features_to_remove
})

display(removed_features_summary.head(40))

In [ ]:
# 10) create selected datasets

X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

assert X_train_selected.shape[1] == X_test_selected.shape[1]
assert X_train_selected.isna().sum().sum() == 0
assert X_test_selected.isna().sum().sum() == 0
assert len(X_train_selected.select_dtypes(exclude="number").columns) == 0
assert len(X_test_selected.select_dtypes(exclude="number").columns) == 0

print("X_train_selected shape:", X_train_selected.shape)
print("y_train shape:", y_train.shape)
print("X_test_selected shape:", X_test_selected.shape)

In [ ]:
# 11) final visual checks

selection_summary = pd.DataFrame({
    "step": [
        "original features",
        "near-constant features removed",
        "correlated features removed",
        "zero-importance features removed",
        "final selected features"
    ],
    "count": [
        X_train.shape[1],
        len(near_constant_features),
        len(features_to_remove_from_correlation),
        len(zero_importance_features),
        len(selected_features)
    ]
})

display(selection_summary)

print("top selected features by importance")

display(
    feature_importance[
        feature_importance["feature"].isin(selected_features)
    ].head(30)
)

print("preview of selected training features")

display(X_train_selected.head().T)

In [ ]:
# The first section imports the required libraries and checks that X_train, y_train, and X_test are available
# from the previous preprocessing and feature engineering notebook.
#
# The second section performs a basic quality check on the features, including missing values, data types,
# and the number of unique values for each feature.
#
# The third section identifies constant and near-constant features, which are unlikely to help the model
# because they contain little or no variation.
#
# The fourth section checks for highly correlated numerical features. Strongly correlated features may provide
# redundant information and can make the model more complex.
#
# The fifth section selects which correlated features should be removed, keeping one feature from each highly
# correlated pair.
#
# The sixth section creates an internal train-validation split using only the training data. This split is used
# only to estimate feature importance and does not involve the test set.
#
# The seventh section trains a Random Forest model on the internal training split to estimate feature importance.
#
# The eighth section displays the most important features according to the baseline Random Forest model.
#
# The ninth section combines the feature selection rules and creates the final list of selected features.
# It removes near-constant features, redundant correlated features, and features with zero model importance.
#
# The tenth section creates X_train_selected and X_test_selected using the same selected feature columns.
# The test set is not used to decide which features to keep; it is only filtered using the choices made from
# the training data.
#
# The eleventh section displays a final summary of the feature selection process and previews the selected
# training features.